In [ ]:
import numpy as np
import os
import scipy
from dotenv import load_dotenv

from rfml_uav.drone_rf.consts import BUI

load_dotenv()

In [ ]:
load_filename = os.path.join(os.getenv("DATA_PATH"), "PSD", "mat")
save_filename = load_filename

In [ ]:
# Initialize empty lists for data and lengths
DATA = []
LN = []

# Loading and concatenating RF data
for bui in BUI:
    # Loading the MATLAB .mat file
    mat_data = scipy.io.loadmat(os.path.join(load_filename, f"{bui}.mat"))
    Data = mat_data['Data']  # Assuming 'Data' is the key in the .mat file
    Data = Data / np.max(Data)  # Normalizing the data
    DATA.append(Data)
    LN.append(Data.shape[1])  # Storing the length of each Data matrix

# Labeling
Label = np.zeros((3, sum(LN)), dtype=int)

# Labeling first row
Label[0, :] = np.concatenate([np.zeros(LN[0]), np.ones(sum(LN[1:])),])

# Labeling second row
Label[1, :] = np.concatenate([np.zeros(LN[0]), 
                               np.ones(sum(LN[1:5])), 
                               np.full(sum(LN[5:9]), 2),
                               np.full(LN[9], 3)])

# Labeling third row
temp = []
for i in range(len(LN)):
    temp.extend([i] * LN[i])
Label[2, :] = np.array(temp)

print(Label)


In [ ]:
# Concatenate the data
DATA = np.hstack(DATA)

# Saving the final data and label to CSV
np.savetxt(os.path.join(save_filename, 'RF_Data.csv'), np.vstack([DATA, Label]), delimiter=',')